# Export Model MobileNetV2 (CornGuard)

Notebook ini mengekspor model klasifikasi 4 kelas penyakit daun jagung (Blight, Common_Rust, Gray_Leaf_Spot, Healthy) ke format .keras, .tflite, dan .onnx beserta file metadata untuk integrasi ke sistem CornGuard.

**1. Upload dan ekstrak dataset**

Cell ini mengunggah dataset bersih corn_dataset_clean.zip (3.636 file) ke Colab, membersihkan sisa file lama agar tidak bentrok, memeriksa bahwa zip tidak rusak, lalu mengekstraknya. Hasilnya empat folder kelas: Blight, Common_Rust, Gray_Leaf_Spot, dan Healthy.

In [4]:
import os, zipfile, shutil
from google.colab import files

# Bersihkan sisa file lama
for f in os.listdir('.'):
    if f.startswith('corn_dataset_clean') and f.endswith('.zip'):
        os.remove(f)
shutil.rmtree('/content/corn_dataset', ignore_errors=True)

# Upload ulang dan tunggu sampai 100%
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print("Nama file:", zip_name, "| Ukuran:", os.path.getsize(zip_name), "byte")

# Ukuran yang benar: 50297980 byte
assert zipfile.is_zipfile(zip_name), "Zip rusak, upload ulang sampai 100%."
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/corn_dataset')
print("Berhasil diekstrak:", os.listdir('/content/corn_dataset'))

Saving corn_dataset_clean.zip to corn_dataset_clean.zip
Nama file: corn_dataset_clean.zip | Ukuran: 50297980 byte
Berhasil diekstrak: ['Common_Rust', 'Gray_Leaf_Spot', 'Blight', 'Healthy']


**2. Training MobileNetV2 dan export .keras, .tflite**

Model dilatih ulang dalam dua tahap. Tahap pertama adalah feature extraction, yaitu bobot MobileNetV2 dibekukan dan hanya lapisan klasifikasi yang dilatih. Tahap kedua adalah fine-tuning, yaitu sebagian lapisan atas MobileNetV2 dibuka dengan learning rate kecil dan class weight untuk menyeimbangkan jumlah data tiap kelas. Model dengan akurasi validasi terbaik dipilih, lalu disimpan ke format .keras dan .tflite beserta labels_dan_metadata.json. Data dibagi 80% training dan 20% validasi.

In [5]:
import os, json, zipfile, pathlib
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight

# 1. Ekstrak dataset (zip sudah ada di Colab dari upload tadi)
root = pathlib.Path('/content/corn_dataset')
data_dir = root / 'data' if (root / 'data').is_dir() else root

IMG_SIZE, BATCH_SIZE, SEED = (224, 224), 32, 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='training',
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset='validation',
    seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE)
class_names = train_ds.class_names
print("Kelas:", class_names)

train_ds = train_ds.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

# 2. Model
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.15),
    layers.RandomZoom(0.15), layers.RandomContrast(0.1)])
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# 3. Tahap 1: feature extraction
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15,
               callbacks=[tf.keras.callbacks.EarlyStopping(
                   monitor='val_loss', patience=4, restore_best_weights=True)])
acc1 = model.evaluate(val_ds, verbose=0)[1]
model.save('model_tahap1.keras')

# 4. Tahap 2: fine-tuning dengan class weight
labels_train = np.concatenate([y.numpy() for _, y in train_ds])
cw = compute_class_weight('balanced', classes=np.unique(labels_train), y=labels_train)
class_weight = {i: float(w) for i, w in enumerate(cw)}

base_model.trainable = True
for layer in base_model.layers[:120]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=len(h1.epoch) + 10,
          initial_epoch=len(h1.epoch), class_weight=class_weight,
          callbacks=[
              tf.keras.callbacks.ModelCheckpoint('model_terbaik.keras',
                  monitor='val_loss', save_best_only=True),
              tf.keras.callbacks.EarlyStopping(
                  monitor='val_loss', patience=4, restore_best_weights=True)])
acc2 = model.evaluate(val_ds, verbose=0)[1]
print(f"Tahap 1: {acc1:.4f} | Setelah fine-tuning: {acc2:.4f}")

# 5. Pakai model terbaik
if acc2 < acc1:
    model = tf.keras.models.load_model('model_tahap1.keras')
    model_path = 'model_tahap1.keras'
else:
    model_path = 'model_terbaik.keras'
print("Model final:", model_path)

# 6. Export
os.makedirs('export_model', exist_ok=True)
model.save('export_model/corn_mobilenetv2.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
with open('export_model/corn_mobilenetv2.tflite', 'wb') as f:
    f.write(converter.convert())

meta = {
    'arsitektur': 'MobileNetV2',
    'kelas': class_names,
    'input_shape': [224, 224, 3],
    'input_dtype': 'float32, nilai piksel 0-255 (preprocessing sudah di dalam model)',
    'output': 'softmax 4 kelas',
    'sumber_model': model_path,
    'akurasi_validasi': round(float(max(acc1, acc2)), 4),
}
with open('export_model/labels_dan_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

for nama in os.listdir('export_model'):
    print(nama, round(os.path.getsize(f'export_model/{nama}') / 1024 / 1024, 2), "MB")

Found 3632 files belonging to 4 classes.
Using 2906 files for training.
Found 3632 files belonging to 4 classes.
Using 726 files for validation.
Kelas: ['Blight', 'Common_Rust', 'Gray_Leaf_Spot', 'Healthy']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 208s 2s/step - accuracy: 0.8245 - loss: 0.4702 - val_accuracy: 0.9270 - val_loss: 0.2104
Epoch 2/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 188s 2s/step - accuracy: 0.9136 - loss: 0.2237 - val_accuracy: 0.9408 - val_loss: 0.1730
Epoch 3/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 195s 2s/step - accuracy: 0.9243 - loss: 0.1835 - val_accuracy: 0.9421 - val_loss: 0.1532
Epoch 4/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.9391 - loss: 0.1688 - val_accuracy: 0.9435 - val_loss: 0.1567
Epoch 5/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - accuracy: 0.9391 - loss: 0.1535 - val_accuracy: 0.9463 - val_loss: 0.1441
Epoch 6/15
91/91 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - accuracy: 0.9432 - loss: 0.1520 - val_accuracy: 0.9601 -

**3. Verifikasi model .keras dan .tflite**

Cell ini menguji model .keras dan .tflite pada data validasi yang sama untuk memastikan konversi tidak merusak hasil prediksi. Akurasi .keras 0,9601 dan .tflite 0,9614, dengan 98,76% prediksi yang sama. Selisih kecil itu wajar karena .tflite dikompresi agar ukurannya hanya sekitar 2,4 MB.

In [7]:
interpreter = tf.lite.Interpreter(model_path='export_model/corn_mobilenetv2.tflite')
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

y_true, p_keras, p_tflite = [], [], []
for images, labels in val_ds:
    probs_k = model.predict(images, verbose=0)
    for img, lab, pk in zip(images.numpy(), labels.numpy(), probs_k):
        interpreter.set_tensor(inp['index'], np.expand_dims(img, 0).astype('float32'))
        interpreter.invoke()
        pt = interpreter.get_tensor(out['index'])[0]
        y_true.append(lab); p_keras.append(np.argmax(pk)); p_tflite.append(np.argmax(pt))

y_true, p_keras, p_tflite = map(np.array, (y_true, p_keras, p_tflite))
print(f"Akurasi .keras : {(y_true == p_keras).mean():.4f}")
print(f"Akurasi .tflite: {(y_true == p_tflite).mean():.4f}")
print(f"Prediksi sama  : {(p_keras == p_tflite).mean():.4f}")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Akurasi .keras : 0.9601
Akurasi .tflite: 0.9614
Prediksi sama  : 0.9876


**4. Konversi ke** ***ONNX***

Model dikonversi ke format ONNX supaya bisa dijalankan di backend memakai onnxruntime tanpa TensorFlow. Sebelum konversi, dibuat model khusus inference tanpa layer augmentasi dan dropout, karena kedua layer itu hanya dipakai saat training dan operasi acaknya tidak didukung ONNX. Bobot model tetap sama, dan prediksinya sudah dicek tidak berbeda.

In [11]:
import shutil

# Model inference: tanpa augmentasi dan dropout, bobot sama dengan model terlatih
base = next(l for l in model.layers if 'mobilenet' in l.name.lower())
dense = model.layers[-1]

inp_layer = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inp_layer)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
infer_model = tf.keras.Model(inp_layer, dense(x))

# Pastikan prediksinya identik dengan model asli
batch = next(iter(val_ds))[0]
selisih = np.abs(model.predict(batch, verbose=0) - infer_model.predict(batch, verbose=0)).max()
print("Selisih maksimum prediksi:", selisih)

# Bersihkan sisa konversi lama, lalu konversi ulang
shutil.rmtree('saved_model_cg', ignore_errors=True)
if os.path.exists('export_model/corn_mobilenetv2.onnx'):
    os.remove('export_model/corn_mobilenetv2.onnx')

infer_model.export('saved_model_cg')
!python -m tf2onnx.convert --saved-model saved_model_cg --output export_model/corn_mobilenetv2.onnx --opset 13

Selisih maksimum prediksi: 0.0
Saved artifact at 'saved_model_cg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_507')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137198459434192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459434000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459433232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459434576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459434768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459434960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459435920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459435728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459436112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137198459434384: TensorSpec(shape=(), dtype=tf.reso

**5. Verifikasi ONNX dan buat model_meta**.**json**

Model ONNX diuji dengan onnxruntime pada data validasi. Hasilnya akurasi 0,9601 dan prediksi 100% sama dengan model .keras. Cell ini juga membuat model_meta.json yang berisi nama dan bentuk input/output, urutan kelas, rentang piksel, dan akurasi, sebagai panduan bagi backend dalam memakai model.

In [13]:
import onnxruntime as ort

sess = ort.InferenceSession('export_model/corn_mobilenetv2.onnx')
in_info, out_info = sess.get_inputs()[0], sess.get_outputs()[0]
print("Input :", in_info.name, in_info.shape, in_info.type)
print("Output:", out_info.name, out_info.shape, out_info.type)

p_onnx = []
for images, labels in val_ds:
    probs = sess.run(None, {in_info.name: images.numpy().astype('float32')})[0]
    p_onnx.extend(np.argmax(probs, axis=1))
p_onnx = np.array(p_onnx)

acc_onnx = float((y_true == p_onnx).mean())
print(f"Akurasi .onnx   : {acc_onnx:.4f}")
print(f"Sama dgn .keras : {(p_keras == p_onnx).mean():.4f}")

model_meta = {
    'nama_model': 'CornGuard MobileNetV2',
    'format': 'onnx',
    'file_model': 'corn_mobilenetv2.onnx',
    'arsitektur': 'MobileNetV2 (transfer learning, ImageNet)',
    'kelas': class_names,
    'jumlah_kelas': len(class_names),
    'input': {'nama': in_info.name, 'shape': [None, 224, 224, 3],
              'dtype': 'float32', 'rentang_piksel': '0-255',
              'catatan': 'preprocessing MobileNetV2 sudah di dalam model'},
    'output': {'nama': out_info.name, 'shape': [None, len(class_names)],
               'keterangan': 'probabilitas softmax per kelas'},
    'akurasi_validasi': round(acc_onnx, 4),
}
with open('export_model/model_meta.json', 'w') as f:
    json.dump(model_meta, f, indent=2, ensure_ascii=False)
print(json.dumps(model_meta, indent=2, ensure_ascii=False))

Input : keras_tensor_507 ['unk__654', 224, 224, 3] tensor(float)
Output: output_0 ['unk__655', 4] tensor(float)
Akurasi .onnx   : 0.9601
Sama dgn .keras : 1.0000
{
  "nama_model": "CornGuard MobileNetV2",
  "format": "onnx",
  "file_model": "corn_mobilenetv2.onnx",
  "arsitektur": "MobileNetV2 (transfer learning, ImageNet)",
  "kelas": [
    "Blight",
    "Common_Rust",
    "Gray_Leaf_Spot",
    "Healthy"
  ],
  "jumlah_kelas": 4,
  "input": {
    "nama": "keras_tensor_507",
    "shape": [
      null,
      224,
      224,
      3
    ],
    "dtype": "float32",
    "rentang_piksel": "0-255",
    "catatan": "preprocessing MobileNetV2 sudah di dalam model"
  },
  "output": {
    "nama": "output_0",
    "shape": [
      null,
      4
    ],
    "keterangan": "probabilitas softmax per kelas"
  },
  "akurasi_validasi": 0.9601
}


**6. Kemas dan unduh export_model.zip**

Seluruh hasil export (.keras, .tflite, .onnx, model_meta.json, dan labels_dan_metadata.json) dikemas menjadi satu file zip lalu diunduh. Cell ini dijalankan paling akhir setelah semua file terbentuk.

In [14]:
import os
import shutil
from google.colab import files

if os.path.exists('export_model'):
    shutil.make_archive('export_model', 'zip', 'export_model')
    files.download('export_model.zip')
else:
    print("Error: Folder 'export_model' tidak ditemukan. Pastikan Anda sudah menjalankan sel pertama (rtxuzox2qcen) untuk membuat dan menyimpan model sebelum mengunduhnya!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Catatan Penggunaan Model

- Urutan kelas output model: Blight, Common_Rust, Gray_Leaf_Spot, Healthy.
- Input gambar berukuran 224x224 piksel dengan nilai piksel 0-255. Jangan dibagi 255, karena preprocessing MobileNetV2 sudah ada di dalam model.
- Nama input pada model ONNX dibaca dari file model_meta.json, bukan ditulis manual, karena bisa berubah jika model dikonversi ulang.
- Keterbatasan: model masih sering menukar kelas Blight dan Gray_Leaf_Spot, sehingga hasil prediksi pada dua kelas ini perlu dipertimbangkan dengan hati-hati.